# Da She's Voice Engine — Colab GPU
GPU-accelerated TTS using XTTS v2.

In [ ]:
# 1. Install Python 3.10 via deadsnakes + TTS
import sys, subprocess

# Install Python 3.10 and pip
!apt-get update -qq 2>&1 | tail -1
!apt-get install -y python3.10 python3.10-dev python3.10-distutils -qq 2>&1 | tail -2
!wget -q https://bootstrap.pypa.io/get-pip.py -O /tmp/get-pip.py
!python3.10 /tmp/get-pip.py -q 2>&1 | tail -1

PY = "/usr/bin/python3.10"
print(f"Python: {subprocess.check_output([PY, '--version']).decode().strip()}")

# Install TTS
r = subprocess.run([PY, "-m", "pip", "install", "-q", "--ignore-installed", "blinker", "TTS"], capture_output=True, text=True, timeout=300)
if r.returncode != 0:
    print(f"TTS failed: {r.stderr[-200:]}")
else:
    print("TTS installed")

subprocess.run([PY, "-m", "pip", "install", "-q", "--ignore-installed", "blinker", "flask", "flask-cors"], capture_output=True)

In [ ]:
# 2. Download speaker and start server
import requests, subprocess, os

r = requests.get("https://qwert.crousia.com/speaker.wav", timeout=30)
r.raise_for_status()
with open("speaker.wav", "wb") as f: f.write(r.content)
print(f"Speaker: {len(r.content)} bytes")

# Server script for Python 3.10
script = r'''
from flask import Flask, request, send_file
from TTS.api import TTS
import torch, tempfile, os, threading
has_gpu = torch.cuda.is_available()
print(f"GPU: {torch.cuda.get_device_name(0) if has_gpu else 'N/A'}")
tts = TTS("tts_models/multilingual/multi-dataset/xtts_v2", gpu=has_gpu)
app = Flask(__name__)
@app.route(\"/health\")
def health(): return {\"status\": \"ok\", \"gpu\": has_gpu}
@app.route(\"/synthesize\", methods=[\"POST\"])
def synth():
    text = request.get_json().get(\"text\", \"\")
    if not text: return {\"error\": \"text required\"}, 400
    fd, path = tempfile.mkstemp(suffix=\".wav\"); os.close(fd)
    tts.tts_to_file(text=text, file_path=path, speaker_wav=\"speaker.wav\", language=\"en\")
    return send_file(path, mimetype=\"audio/wav\")
threading.Thread(target=lambda: app.run(host=\"0.0.0.0\", port=5000, debug=False), daemon=True).start()
print(\"Server on :5000\")
import time
while True: time.sleep(60)
'''

proc = subprocess.Popen(["/usr/bin/python3.10", "-c", script],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
print("Server started")

In [ ]:
# 3. Cloudflare Tunnel
import subprocess, time, re
!curl -sL https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -o /usr/local/bin/cloudflared && chmod +x /usr/local/bin/cloudflared
p = subprocess.Popen(["cloudflared", "tunnel", "--url", "http://127.0.0.1:5000"],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
url = None
for _ in range(30):
    time.sleep(1)
    out = (p.stdout.read(4096) if p.stdout else "")
    m = re.search(r'https://[a-z0-9-]+\\.trycloudflare\\.com', out)
    if m: url = m.group(0); break
if url: print(f"ENDPOINT: {url}")
else: print("No tunnel URL")
while True: time.sleep(60)